In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
功能：比较 asr/ancestrals 下所有 .fasta 文件中的序列与 query.fasta 的突变位数。
输出：mutation_summary.csv，包含 file_id, mutations, sequence。
路径已固定为：E:\工作\汇报4\ASR\87x8da
"""

import os
import sys
import csv
from Bio import SeqIO


def read_reference(query_path):
    """读取参考序列（query.fasta 中的第一条序列）"""
    try:
        for record in SeqIO.parse(query_path, "fasta"):
            return str(record.seq)
    except FileNotFoundError:
        sys.exit(f"错误：找不到参考序列文件 {query_path}")
    raise ValueError("参考序列文件为空或没有有效序列")


def count_mutations(ref_seq, test_seq):
    """统计两个等长序列的不同位点数（逐位比较）"""
    if len(ref_seq) != len(test_seq):
        raise ValueError(f"长度不一致: ref={len(ref_seq)}, test={len(test_seq)}")
    return sum(1 for a, b in zip(ref_seq, test_seq) if a != b)


def process_ancestrals(anc_dir, ref_seq):
    """遍历 anc_dir 下所有 .fasta 文件，提取每条序列并统计突变数"""
    results = []
    if not os.path.isdir(anc_dir):
        sys.exit(f"错误：祖先序列目录不存在 {anc_dir}")

    for filename in os.listdir(anc_dir):
        if filename.lower().endswith('.fasta'):
            filepath = os.path.join(anc_dir, filename)
            for record in SeqIO.parse(filepath, "fasta"):
                seq = str(record.seq)
                record_id = record.id if record.id else filename
                try:
                    mut_count = count_mutations(ref_seq, seq)
                except ValueError as e:
                    print(f"跳过 {filename} 中的 {record_id}: {e}")
                    continue
                results.append({
                    'file_id': record_id,
                    'mutations': mut_count,
                    'sequence': seq
                })
    return results


def main():
    # ========== ✅ 路径已直接修改为你的实际位置 ==========
    base_dir = r'E:\工作\汇报4\ASR\87x8da'
    # ==================================================

    if not os.path.isdir(base_dir):
        sys.exit(f"错误：找不到目录 {base_dir}\n请确认路径是否正确。")

    query_path = os.path.join(base_dir, 'query.fasta')
    ancest_dir = os.path.join(base_dir, 'asr', 'ancestrals')
    output_csv = os.path.join(base_dir, 'mutation_summary.csv')

    # 检查 query.fasta 是否存在
    if not os.path.isfile(query_path):
        sys.exit(f"错误：找不到参考序列文件 {query_path}")

    ref_seq = read_reference(query_path)
    print(f"✅ 参考序列长度: {len(ref_seq)}")

    results = process_ancestrals(ancest_dir, ref_seq)

    if not results:
        print("⚠️ 警告：未找到任何有效的祖先序列。请检查 asr/ancestrals 目录下是否有 .fasta 文件。")
        return

    results.sort(key=lambda x: x['mutations'])

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['file_id', 'mutations', 'sequence'])
        writer.writeheader()
        writer.writerows(results)

    print(f"✅ 完成！共处理 {len(results)} 个序列，结果已保存至：{output_csv}")
    print("（按突变数从小到大排序）")


if __name__ == '__main__':
    main()

✅ 参考序列长度: 298
跳过 ancestral_151.fasta 中的 ancestral_151: 长度不一致: ref=298, test=279
跳过 ancestral_152.fasta 中的 ancestral_152: 长度不一致: ref=298, test=259
跳过 ancestral_153.fasta 中的 ancestral_153: 长度不一致: ref=298, test=272
跳过 ancestral_154.fasta 中的 ancestral_154: 长度不一致: ref=298, test=285
跳过 ancestral_155.fasta 中的 ancestral_155: 长度不一致: ref=298, test=244
跳过 ancestral_156.fasta 中的 ancestral_156: 长度不一致: ref=298, test=301
跳过 ancestral_157.fasta 中的 ancestral_157: 长度不一致: ref=298, test=290
跳过 ancestral_158.fasta 中的 ancestral_158: 长度不一致: ref=298, test=296
跳过 ancestral_159.fasta 中的 ancestral_159: 长度不一致: ref=298, test=311
跳过 ancestral_160.fasta 中的 ancestral_160: 长度不一致: ref=298, test=271
跳过 ancestral_161.fasta 中的 ancestral_161: 长度不一致: ref=298, test=314
跳过 ancestral_162.fasta 中的 ancestral_162: 长度不一致: ref=298, test=277
跳过 ancestral_163.fasta 中的 ancestral_163: 长度不一致: ref=298, test=295
跳过 ancestral_164.fasta 中的 ancestral_164: 长度不一致: ref=298, test=285
跳过 ancestral_165.fasta 中的 ancestral_165: 长度不一致: ref=298, test=

<>:7: SyntaxWarning: invalid escape sequence '\A'
<>:7: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_10108\1601883833.py:7: SyntaxWarning: invalid escape sequence '\A'
  路径已固定为：E:\工作\汇报4\ASR\87x8da


In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
功能：检验候选序列是否满足设定的突变约束。
输入：query.fasta（原始序列），候选序列（手动粘贴在 CANDIDATE_SEQ 中）。
输出：打印详细检验报告。
"""

import os
import sys
from Bio import SeqIO

# ========== 路径设置 ==========
BASE_DIR = r'E:\工作\汇报4\ASR\87x8da'          # 你的数据目录
QUERY_FASTA = os.path.join(BASE_DIR, 'query.fasta')

# 候选序列（请在此粘贴你要检验的序列，或从文件读取）
CANDIDATE_SEQ = """MFLRRLFGVVAALAVLAHAAAAPAPLERRDVSSDLLDQLDLFAQYSAAAYCSSNLNSAGTTVTCSAGNCPLVEAANTTTLYEFDDTSSYGDTTGFLAVDSTNKLIVLSFRGSSSLENWIADLDFGLVDASSICNGCEVHKGFWESWNSVADTLTSKIESAVNAYPDYSLVFTGHSLGGALATLGATVLRNAGYNVDLYTYGCPRVGNTALADYITNQSSGSNYRVTHTDDVVPKLPPRLLGYSQPSPEYWITSGNDVTVTTSDIEVIEGVDSTAGNAGTAAASIDAHRWYFINISACS"""

# ==============================================

# ----- 约束规则（全部使用 full sequence 位置，1-based） -----

# 1. 硬保留（必须与 query 一致，不能发生突变）
HARD_CONSTRAINTS = {
    175: 'S',   # 催化三联体
    230: 'D',
    287: 'H',
    288: 'R',   # PLA2功能
    50:  'Y',   # 与sn-2结合
    112: 'S',   # Oxyanion hole
    121: 'D',   # Stabilises oxyanion hole
}

# 2. 必须引入的突变（原始 -> 目标，候选必须为目标氨基酸）
REQUIRED_MUTATIONS = {
    123: ('D', 'N'),   # D123N
}

# 3. 可优化突变（非强制，若发生则提示，支持多个目标用 '/' 分隔）
OPTIONAL_MUTATIONS = {
    176: ('Y', 'L'),   # 原注释“无此突变”，但保留
    113: ('S', 'R'),
    114: ('D', 'T'),
    238: ('E', 'K'),
    296: ('E', 'R/Q'), # 可变为 R 或 Q
}

# 4. 必须避免的突变（若候选变为这些氨基酸则报错，支持多个目标用 '/' 分隔）
FORBIDDEN_MUTATIONS = {
    84:  ('D', 'S'),      # D84S
    91:  ('D', 'S/N'),    # D91S 或 D91N
    243: ('H', 'K'),      # H243K
    289: ('W', 'S/V'),    # W289S 或 W289V
    294: ('I', 'T'),      # I294T
    295: ('S', 'R'),      # S295R
}

# ========== 工具函数 ==========

def read_query(query_path):
    """读取参考序列（query.fasta）"""
    if not os.path.isfile(query_path):
        sys.exit(f"错误：找不到参考序列文件 {query_path}")
    for record in SeqIO.parse(query_path, "fasta"):
        return str(record.seq)
    raise ValueError("参考序列文件为空或没有有效序列")

def get_mutations(ref, cand):
    """比较两个等长序列，返回差异位点字典 {位置: (ref_aa, cand_aa)}"""
    if len(ref) != len(cand):
        raise ValueError(f"序列长度不一致: ref={len(ref)}, cand={len(cand)}")
    mutations = {}
    for i, (a, b) in enumerate(zip(ref, cand), start=1):
        if a != b:
            mutations[i] = (a, b)
    return mutations

def check_constraints(mutations):
    """检查所有约束，返回报告字符串"""
    report = []
    report.append("="*60)
    report.append("突变检验报告")
    report.append("="*60)

    # ---- 1. 硬保留检查 ----
    hard_passed = True
    for pos, req_aa in HARD_CONSTRAINTS.items():
        if pos in mutations:
            ref_aa, cand_aa = mutations[pos]
            if cand_aa != req_aa:
                hard_passed = False
                report.append(f"❌ 硬约束失败：位置 {pos} 要求 {req_aa}，候选为 {cand_aa}（原始 {ref_aa}）")
            else:
                # 这种情况理论上不会发生，因为如果突变后仍等于req_aa，说明突变前后相同，属于不可能
                report.append(f"⚠️ 位置 {pos} 发生突变但依然等于 {req_aa}，可能是序列录入错误")
        else:
            report.append(f"✅ 硬约束通过：位置 {pos} 未突变，保持为 {req_aa}")
    report.append("✅ 所有硬约束均通过" if hard_passed else "❌ 硬约束未通过")

    # ---- 2. 必须引入的突变 ----
    req_passed = True
    for pos, (ref_aa, target_aa) in REQUIRED_MUTATIONS.items():
        if pos in mutations:
            ref, cand = mutations[pos]
            if cand == target_aa:
                report.append(f"✅ 必须引入成功：位置 {pos} {ref} → {cand}（目标 {target_aa}）")
            else:
                req_passed = False
                report.append(f"❌ 必须引入失败：位置 {pos} 应为 {target_aa}，候选为 {cand}（原始 {ref}）")
        else:
            req_passed = False
            report.append(f"❌ 必须引入失败：位置 {pos} 未突变，仍为 {ref_aa}，需变为 {target_aa}")
    report.append("✅ 所有必须引入均满足" if req_passed else "❌ 必须引入未满足")

    # ---- 3. 可优化突变（仅提示） ----
    report.append("\n可优化突变检查：")
    for pos, (ref_aa, target_str) in OPTIONAL_MUTATIONS.items():
        target_list = target_str.split('/')   # 支持多个目标，如 'R/Q'
        if pos in mutations:
            ref, cand = mutations[pos]
            if cand in target_list:
                report.append(f"  ✅ 已发生可优化突变：位置 {pos} {ref} → {cand}（推荐 {target_str}）")
            else:
                report.append(f"  ℹ️  位置 {pos} 发生突变 {ref} → {cand}，但未达到推荐目标 {target_str}")
        else:
            report.append(f"  ⚪ 位置 {pos} 未突变（原始 {ref_aa}），可考虑优化为 {target_str}")

    # ---- 4. 必须避免的突变 ----
    forbidden_violations = []
    for pos, (ref_aa, forbidden_str) in FORBIDDEN_MUTATIONS.items():
        forbidden_list = forbidden_str.split('/')  # 支持多个，如 'S/N'
        if pos in mutations:
            ref, cand = mutations[pos]
            if ref == ref_aa and cand in forbidden_list:
                forbidden_violations.append(f"位置 {pos} {ref} → {cand}（禁止突变）")
    if forbidden_violations:
        report.append("❌ 存在禁止突变：")
        for v in forbidden_violations:
            report.append(f"   {v}")
    else:
        report.append("✅ 未发现禁止突变")

    # ---- 5. 所有差异位点汇总 ----
    report.append("\n所有差异位点汇总：")
    if mutations:
        for pos in sorted(mutations.keys()):
            ref, cand = mutations[pos]
            report.append(f"  {pos}: {ref} → {cand}")
    else:
        report.append("  无差异（候选序列与 query 完全一致）")

    report.append("="*60)
    return "\n".join(report)

def main():
    # 读取参考序列
    query_seq = read_query(QUERY_FASTA)
    print(f"Query 长度: {len(query_seq)}")

    # 候选序列（去除空白和换行）
    cand_seq = ''.join(CANDIDATE_SEQ.split())
    print(f"候选长度: {len(cand_seq)}")

    if len(query_seq) != len(cand_seq):
        print(f"错误：长度不一致，无法比较。")
        return

    mutations = get_mutations(query_seq, cand_seq)
    report = check_constraints(mutations)
    print(report)

if __name__ == '__main__':
    main()

Query 长度: 298
候选长度: 298
突变检验报告
✅ 硬约束通过：位置 175 未突变，保持为 S
✅ 硬约束通过：位置 230 未突变，保持为 D
✅ 硬约束通过：位置 287 未突变，保持为 H
✅ 硬约束通过：位置 288 未突变，保持为 R
✅ 硬约束通过：位置 50 未突变，保持为 Y
✅ 硬约束通过：位置 112 未突变，保持为 S
✅ 硬约束通过：位置 121 未突变，保持为 D
✅ 所有硬约束均通过
❌ 必须引入失败：位置 123 应为 N，候选为 D（原始 N）
❌ 必须引入未满足

可优化突变检查：
  ✅ 已发生可优化突变：位置 176 Y → L（推荐 L）
  ⚪ 位置 113 未突变（原始 S），可考虑优化为 R
  ℹ️  位置 114 发生突变 D → S，但未达到推荐目标 T
  ℹ️  位置 238 发生突变 E → R，但未达到推荐目标 K
  ℹ️  位置 296 发生突变 E → A，但未达到推荐目标 R/Q
✅ 未发现禁止突变

所有差异位点汇总：
  6: E → L
  9: A → V
  14: S → A
  21: P → A
  26: M → L
  27: Q → E
  31: I → V
  34: T → D
  35: V → L
  38: N → Q
  39: I → L
  55: I → L
  56: E → N
  58: T → A
  62: L → V
  65: D → S
  66: V → A
  76: G → N
  77: A → T
  80: I → L
  81: D → Y
  92: P → T
  96: I → L
  100: P → S
  103: E → K
  114: D → S
  116: S → E
  123: N → D
  127: T → V
  128: S → D
  129: V → A
  134: D → N
  138: M → V
  143: Y → W
  145: A → S
  147: E → N
  148: V → S
  149: I → V
  153: I → L
  157: V → I
  159: A → S
  162: S → N
  163: S → A
  168: 